# Create Recipe Programatically

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


In [ ]:
from femtum_sdk.api.contracts.v1.instruments.health_pb2 import HealthRequest
from femtum_sdk.api import DatastoreApi, InstrumentApi

datastore_host = "localhost:5208"
instrument_host = "localhost:5210"

with DatastoreApi(host=datastore_host) as datastore_api:
    datastore_health_response = datastore_api.health.GetHealth(HealthRequest())
    print("Datastore Health:")
    print(datastore_health_response)

with InstrumentApi(host=instrument_host) as instrument_api:
    instrument_health_response = instrument_api.health.GetHealth(HealthRequest())
    print("Instrument Health:")
    print(instrument_health_response)


## Create Single Point Recipe

First get camera size to be assigned to recipe

In [ ]:
from femtum_sdk.api.contracts.v1.dto.camera_pb2 import CameraInfo
from femtum_sdk.api.contracts.v1.instruments.camera_pb2 import ReadCameraInfoRequest, ResetROIRequest

with InstrumentApi(host=instrument_host) as instrument_api:
    instrument_api.laser_head_camera.ResetROI(ResetROIRequest())
    camera_info: CameraInfo = instrument_api.laser_head_camera.ReadCameraInfo(ReadCameraInfoRequest())
    context_width: int = camera_info.max_frame_size.width
    context_height: int = camera_info.max_frame_size.height

print(f"{context_width}, {context_height}")

In [ ]:
from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import Attribute, Geometry, Point, Shape
from femtum_sdk.api.contracts.v1.dto.recipe_pb2 import LaserParameter
from femtum_sdk.api.contracts.v1.storage.recipe_pb2 import CreateRecipeRequest

with DatastoreApi(host=datastore_host) as datastore_api:
    single_point_recipe = datastore_api.recipe.Create(CreateRecipeRequest(
        name="Single Point Marking Path", 
        geometries=[
            Geometry(
                type="laser_marking",
                attributes=[
                    Attribute(
                        key="marking_type",
                        value="single_pulse",
                    )
                ],
                shape=Shape(
                    point=Point(
                        x=context_width/2,
                        y=context_height/2,
                    )
                ),
            )
        ],
        attributes=[
            Attribute(
                key="context.width",
                value=f"{context_width}",
            ),
            Attribute(
                key="context.height",
                value=f"{context_height}",
            ),
        ],
        laser_parameters=[
            LaserParameter(key="pulse_train_laser_triggering_frequency_khz", value="2732", unit="kHz"),
            LaserParameter(key="single_pulse_laser_triggering_frequency_khz", value="2732", unit="kHz"),
        ]
    ))


## Waveguide

In [ ]:
from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import (
    Attribute,
    Coordinate,
    Geometry,
    GeometryDistanceUnit,
    LineString,
    Shape,
)
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import PixelMeasureUnit
from femtum_sdk.api.contracts.v1.dto.recipe_pb2 import LaserParameter
from femtum_sdk.api.contracts.v1.storage.recipe_pb2 import CreateRecipeRequest

with DatastoreApi(host=datastore_host) as datastore_api:
    waveguide_recipe = datastore_api.recipe.Create(CreateRecipeRequest(
        name="Single Linestring Waveguide", 
        geometries=[
            Geometry(
                type="waveguide",
                attributes=[
                    Attribute(
                        key="marking_type",
                        value="single_pulse",
                    )
                ],
                shape=Shape(
                    line_string=LineString(
                        path=[
                            Coordinate(
                                x=context_width/2,
                                y=context_height/2,
                            ),
                            Coordinate(
                                x=context_width/2 + 100,
                                y=context_height/2 + 100,
                            )
                        ],
                        unit=GeometryDistanceUnit(pixel=PixelMeasureUnit.Pixel),
                    )
                ),
            )
        ],
        attributes=[
            Attribute(
                key="context.width",
                value=f"{context_width}",
            ),
            Attribute(
                key="context.height",
                value=f"{context_height}",
            ),
        ],
        laser_parameters=[
            LaserParameter(key="pulse_train_laser_triggering_frequency_khz", value="2732", unit="kHz"),
            LaserParameter(key="single_pulse_laser_triggering_frequency_khz", value="2732", unit="kHz"),
        ]
    ))